In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.show=lambda *a,**k:None
import warnings
warnings.filterwarnings('ignore')


In [2]:
import numpy as np
from config import H,W,FWC,GAIN,RN,BIAS,R_AIRY_PX,PSF_FINE,OVERSAMPLE,JITTER_MAX_SPX
from psf import PSF, annotated_heatmap
from background import Background
from imagelag import ImageLag
from filter import Filter


In [3]:
psf = PSF(jitter_spx=2, jitter_spy=2, A=1.0)
k_fine, tmpl = psf.plot()


In [4]:
A   = 100.0
lag = ImageLag()
print(lag)

# ── Step 1: PSF 模板（使用 psf 存储的 jitter）────────────────────
tmpl_A = psf.make_template(A=A)
th, tw = tmpl_A.shape
print(f"\nCamera PSF: {th}×{tw}, sum={tmpl_A.sum():.2f}, max={tmpl_A.max():.2f}")

# ── Step 2: 嵌入 patch（右侧留 T 列给拖尾）───────────────────────
patch_w = tw + lag.T + 1
patch   = np.zeros((th, patch_w))
patch[:, :tw] = tmpl_A

# ── Step 3: lag forward ──────────────────────────────────────────
out, trapped, remaining, lag_only = lag.apply(patch, return_parts=True)
f_mean = float(trapped[trapped > 0.01].mean()) if (trapped > 0.01).any() else 1.0

sum_in, sum_out = patch.sum(), out.sum()
print(f"电荷守恒: Σin={sum_in:.2f}  Σout={sum_out:.2f}  "
      f"差={sum_in-sum_out:+.3f} ({(sum_in-sum_out)/sum_in*100:.3f}%)")

# ── 绘图 ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 9))

ax0 = plt.subplot2grid((3, 6), (0, 0), colspan=2)
annotated_heatmap(ax0, tmpl_A, f'Camera PSF  {th}×{tw}\nsum={tmpl_A.sum():.2f}',
                  fmt='{:.2f}', fontsize=8)

ax1 = plt.subplot2grid((3, 6), (0, 2), colspan=2)
Qs = np.linspace(0, max(300, A*1.2), 200)
ax1.plot(Qs, lag.f_trap(Qs), 'b-', lw=2, label=r'$f(Q)=N_t(1-e^{-Q/Q_c})$')
ax1.axhline(lag.Nt, color='r', ls='--', lw=1, label=f'$N_t={lag.Nt:.1f}$')
ax1.axvline(lag.Qc, color='g', ls='--', lw=1, label=f'$Q_c={lag.Qc:.1f}$')
ax1.scatter(tmpl_A.flatten(), lag.f_trap(tmpl_A.flatten()),
            c='orange', s=20, alpha=0.7, zorder=5, label='PSF pixels')
ax1.set_xlabel('Q [DN]'); ax1.set_ylabel('f(Q) [DN]')
ax1.set_title('静态非线性'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2 = plt.subplot2grid((3, 6), (0, 4), colspan=2)
t_arr = np.arange(1, lag.T + 1)
for f_val, ls in [(5, ':'), (20, '--'), (f_mean, '-'), (60, '-.')]:
    h_val = lag.kernel(f_val).flatten()[1:]
    A1_val = lag.A1max * (1 - np.exp(-f_val / lag.Fc1))
    ax2.plot(t_arr, h_val, ls, lw=1.5, label=f'f={f_val:.0f}  A1={A1_val:.2f}')
ax2.set_xlabel('τ (frames)'); ax2.set_ylabel('h(τ)')
ax2.set_title('双指数核 h(t; F)')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

ax3 = plt.subplot2grid((3, 6), (1, 0), colspan=3)
annotated_heatmap(ax3, trapped,
                  f'trapped = f(Q)\nsum={trapped.sum():.2f}', fmt='{:.2f}', fontsize=7)

ax4 = plt.subplot2grid((3, 6), (1, 3), colspan=3)
annotated_heatmap(ax4, remaining,
                  f'remaining = Q − f(Q)\nsum={remaining.sum():.2f}', fmt='{:.2f}', fontsize=7)

ax5 = plt.subplot2grid((3, 6), (2, 0), colspan=3)
annotated_heatmap(ax5, lag_only,
                  f'lag tail\nsum={lag_only.sum():.2f}  max={lag_only.max():.2f}',
                  fmt='{:.2f}', fontsize=7)

ax6 = plt.subplot2grid((3, 6), (2, 3), colspan=3)
annotated_heatmap(ax6, out,
                  f'out = remaining + lag\nsum={out.sum():.2f}',
                  fmt='{:.2f}', fontsize=7)

n_cam = psf.psf_fine // psf.oversample
fig.suptitle(
    f'PSF + ImageLag forward chain  (A={A})\n'
    f'PSF: {psf.psf_fine}×{psf.psf_fine} fine → {n_cam}×{n_cam} camera px  '
    f'jitter=({psf.jitter_spx},{psf.jitter_spy})    │    '
    f'Lag: Nt={lag.Nt}  Qc={lag.Qc}  α1={lag.alpha1}  α2={lag.alpha2}',
    fontsize=10, y=0.995)

plt.tight_layout()
plt.savefig('psf_lag_integrated.png', dpi=120, bbox_inches='tight')
plt.show()


ImageLag(Nt=62.63, Qc=85.4)
  slow: α1=0.7779  τ1=3.98 帧
  fast: α2=0.2344  τ2=0.69 帧
  A1(F) = 1.0 · (1−exp(−F/78.5))

Camera PSF: 4×4, sum=99.87, max=32.00
电荷守恒: Σin=99.87  Σout=99.87  差=-0.000 (-0.000%)


In [5]:
A_SHOW = [20, 100, 150]
cx, cy = W // 2, H // 2
half   = 8

bg    = Background(rn=1, bias=5, rng=np.random.RandomState(42))
psf = PSF(jitter_spx=0, jitter_spy=0, A=1.0)
noise = bg.sample()
bg.plot(noise, title='Background noise (rn=1, bias=5)', figsize=(7, 5))
plt.show()
psf.plot(figsize=(17, 4))
for A in A_SHOW:
    signal = np.zeros((H, W))
    psf.stamp(signal, cx, cy, A=A)
    out_signal, trapped, remaining, lag_only = lag.apply(signal, return_parts=True)
    frame = signal + noise
    out   = out_signal + noise

    sl = (slice(cy-half, cy+half), slice(cx-half, cx+half))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    annotated_heatmap(axes[0], frame[sl],
                      f'Before lag  A={A}  ({half*2}×{half*2} crop)', fmt='{:.1f}', fontsize=7)
    annotated_heatmap(axes[1], out[sl],
                      f'After lag  (remaining + lag tail + noise)', fmt='{:.1f}', fontsize=7)
    fig.suptitle(f'A={A} ADC', fontsize=11)
    plt.tight_layout()
    plt.show()


In [6]:
# ── Filter test: box SNR — no-lag vs with-lag ─────────────────────────────
A_TEST = 35.0
CX, CY = W // 2, H // 2
half   = 4
BIAS=10
RN=2
jitter_spx, jitter_spy = 0, 0
BOX_N, BOX_M = 2, 5   # ← 只改这里

psf_ft  = PSF(jitter_spx=jitter_spx, jitter_spy=jitter_spy, A=A_TEST)
psf_ft.plot(figsize=(12, 4))
tmpl_ft = psf_ft.make_template(A=1.0)
flt_    = Filter(rn=RN, bias=BIAS)
lag_    = ImageLag()
bg_ft   = Background(rn=RN, bias=BIAS, rng=np.random.RandomState(41))

noise_ = bg_ft.sample()
sig_   = np.zeros((H, W))
psf_ft.stamp(sig_, CX, CY)

frame_nl = sig_              + noise_
frame_wl = lag_.apply(sig_)  + noise_

# ── 全帧 box_map（2×2 均匀核）────────────────────────────────────────────
bmap_nl = flt_.box_map(frame_nl, n=BOX_N, m=BOX_M)
bmap_wl = flt_.box_map(frame_wl, n=BOX_N, m=BOX_M)

# ── Heatmap: frame + box_map，无lag与有lag并排 ───────────────────────────
sl  = (slice(CY - half, CY + half), slice(CX - half, CX + half + lag_.T // 2))
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
annotated_heatmap(axes[0, 0], frame_nl[sl], f'No-lag frame  A={A_TEST}',  fmt='{:.1f}', fontsize=7)
annotated_heatmap(axes[0, 1], bmap_nl[sl],  'box_map (2×2)  no-lag',      fmt='{:.1f}', fontsize=7)
annotated_heatmap(axes[1, 0], frame_wl[sl], 'With-lag frame',             fmt='{:.1f}', fontsize=7)
annotated_heatmap(axes[1, 1], bmap_wl[sl],  'box_map (2×2)  with-lag',    fmt='{:.1f}', fontsize=7)
plt.tight_layout()
plt.show()

# ── Signal & Noise ────────────────────────────────────────────────────────
s_nl = bmap_nl.max()
s_wl = bmap_wl.max()

sigma_th     = flt_.sigma_theory(n=BOX_N, m=BOX_M)
sigma_nl_emp = flt_.sigma_empirical(bmap_nl, CX, CY)
sigma_wl_emp = flt_.sigma_empirical(bmap_wl, CX, CY)

# ── Table ─────────────────────────────────────────────────────────────────
hdr = f"{'Case':20s}  {'signal':>8}  {'sigma_th':>9}  {'sigma_emp':>10}  {'SNR_th':>8}  {'SNR_emp':>8}"
print(hdr)
print('-' * len(hdr))
for name, s, s_emp in [('Box (no lag)',   s_nl, sigma_nl_emp),
                        ('Box (with lag)', s_wl, sigma_wl_emp)]:
    print(f"{name:20s}  {s:8.2f}  {sigma_th:9.4f}  {s_emp:10.4f}"
          f"  {s/sigma_th:8.2f}  {s/s_emp:8.2f}")

print(f"\nSNR degradation (lag vs no-lag): {(s_wl/sigma_wl_emp) / (s_nl/sigma_nl_emp):.3f}x")


Case                    signal   sigma_th   sigma_emp    SNR_th   SNR_emp
-------------------------------------------------------------------------
Box (no lag)             37.77     6.3246      6.2824      5.97      6.01
Box (with lag)           29.12     6.3246      6.2824      4.60      4.64

SNR degradation (lag vs no-lag): 0.771x


In [7]:
# ── Matched filter kernel: PSF full 4×4 rows + lag tail ──────────────────
# PSF 4×4 stamped at (CX, CY):  spans rows [CY-2, CY+2), cols [CX-2, CX+2)
# Lag tail extends rightward from the PSF right edge.
#
# Template built at A=A_TEST to match the operating amplitude.
# MF signal = A_TEST × ‖tmpl_lag‖.
#
# LAG_COLS: columns to keep after the PSF right outer edge (CX+1 inclusive).
# 0 = PSF 4×4 only.  Useful range: 1 … lag_.T (=20).
LAG_COLS = 5   # ← adjust here

TMPL_JX, TMPL_JY = 0, 0   # ← 改这里

sig_tmpl = np.zeros((H, W))
psf_MF = PSF(jitter_spx=TMPL_JX, jitter_spy=TMPL_JY, A=A_TEST)
psf_MF.stamp(sig_tmpl, CX, CY)
psf_MF.plot(figsize=(12, 4))

lr = lag_.apply(sig_tmpl)

# Row range: full PSF height (4 rows)
r0, r1 = CY - 1, CY + 1               # 4 rows
# Col range: PSF left outer edge … PSF right outer edge + LAG_COLS
c0, c1 = CX - 1, CX + 1 + LAG_COLS   # 4 PSF cols + LAG_COLS

tmpl_lag  = lr[r0:r1, c0:c1]
tmpl_norm = tmpl_lag / np.linalg.norm(tmpl_lag)

energy_frac = tmpl_lag.sum() / sig_tmpl.sum()
print(f"Template shape  : {tmpl_lag.shape}  ({r1-r0} rows × {c1-c0} cols)")
print(f"tmpl_lag.sum()  = {tmpl_lag.sum():.4f}  (energy fraction captured: {energy_frac*100:.1f}%)")
print(f"||tmpl||        = {np.linalg.norm(tmpl_lag):.4f}")
print(f"SNR_theory (A={A_TEST:.0f}) = ||tmpl|| / RN = "
      f"{np.linalg.norm(tmpl_lag) / RN:.2f}")

fig, axes = plt.subplots(2, 1, figsize=(max(8, c1-c0), 4))
annotated_heatmap(axes[0], tmpl_lag,
                  f'Raw kernel  shape={tmpl_lag.shape}  sum={tmpl_lag.sum():.3f}  '
                  f'(A={A_TEST:.0f} input, {energy_frac*100:.1f}% PSF energy)',
                  fmt='{:.4f}', fontsize=8)
annotated_heatmap(axes[1], tmpl_norm,
                  'Normalized kernel  ||tmpl_norm||=1',
                  fmt='{:.4f}', fontsize=8)
plt.tight_layout()
plt.show()


Template shape  : (2, 7)  (2 rows × 7 cols)
tmpl_lag.sum()  = 27.6682  (energy fraction captured: 79.1%)
||tmpl||        = 8.3052
SNR_theory (A=35) = ||tmpl|| / RN = 4.15


In [8]:
# ── 应用 MF 到图像，计算 SNR ─────────────────────────────────────────────
resp_mf_nl = flt_.matched(frame_nl - flt_.bias, tmpl_lag)
resp_mf_wl = flt_.matched(frame_wl - flt_.bias, tmpl_lag)

# ── Heatmap（局部裁剪，与 box 一致）─────────────────────────────────────
sl_mf = (slice(CY - half, CY + half), slice(CX - half, CX + half + lag_.T // 2))
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
annotated_heatmap(axes[0], resp_mf_nl[sl_mf], 'MF response  (no-lag frame)',   fmt='{:.2f}', fontsize=7)
annotated_heatmap(axes[1], resp_mf_wl[sl_mf], 'MF response  (with-lag frame)', fmt='{:.2f}', fontsize=7)
plt.tight_layout()
plt.show()

# ── SNR 表格 ──────────────────────────────────────────────────────────────
s_mf_nl = resp_mf_nl.max()
s_mf_wl = resp_mf_wl.max()
sigma_mf_th     = flt_.rn
sigma_mf_nl_emp = flt_.sigma_empirical(resp_mf_nl, CX, CY)
sigma_mf_wl_emp = flt_.sigma_empirical(resp_mf_wl, CX, CY)

hdr = f"{'Case':28s}  {'signal':>8}  {'sigma_th':>9}  {'sigma_emp':>10}  {'SNR_th':>8}  {'SNR_emp':>8}"
print(hdr)
print('-' * len(hdr))
for name, s, s_emp in [
    ('MF lag (no-lag frame)',   s_mf_nl, sigma_mf_nl_emp),
    ('MF lag (with-lag frame)', s_mf_wl, sigma_mf_wl_emp),
]:
    print(f"{name:28s}  {s:8.2f}  {sigma_mf_th:9.4f}  {s_emp:10.4f}"
          f"  {s/sigma_mf_th:8.2f}  {s/s_emp:8.2f}")


Case                            signal   sigma_th   sigma_emp    SNR_th   SNR_emp
---------------------------------------------------------------------------------
MF lag (no-lag frame)            12.69     2.0000      1.9818      6.35      6.40
MF lag (with-lag frame)           8.77     2.0000      1.9818      4.39      4.43


In [9]:
# ── Monte Carlo: Box vs MF-lag  A∈[80,100]  jitter∈[-10,10] fine px ───────
N_MC   = 200
A_LO, A_HI   = 35, 100.0
JIT_MAX      = 5          # fine pixels, both x and y

# MF kernel: full 4-row PSF + complete lag tail (no-jitter reference)
_sig_t = np.zeros((H, W))
PSF(A=A_TEST).stamp(_sig_t, CX, CY)
_lr_t  = lag_.apply(_sig_t)
tmpl_mc = _lr_t[CY-2:CY+2, CX-2:CX+2+lag_.T]   # shape (4, T+4)
print(f"MC kernel shape: {tmpl_mc.shape}  ||tmpl||={np.linalg.norm(tmpl_mc):.4f}")

sigma_box_th = flt_.sigma_theory(n=BOX_N, m=BOX_M)
sigma_mf_th  = flt_.rn

rng_mc = np.random.RandomState(0)
snr = {k: [] for k in ['box_nl', 'box_wl', 'mf_nl', 'mf_wl']}

for _ in range(N_MC):
    A_i  = rng_mc.uniform(A_LO, A_HI)
    jx_i = rng_mc.uniform(-JIT_MAX, JIT_MAX)
    jy_i = rng_mc.uniform(-JIT_MAX, JIT_MAX)

    sig_i   = np.zeros((H, W))
    PSF(jitter_spx=jx_i, jitter_spy=jy_i, A=A_i).stamp(sig_i, CX, CY)
    noise_i = BIAS + rng_mc.normal(0.0, RN, (H, W))

    frame_nl_i = sig_i              + noise_i
    frame_wl_i = lag_.apply(sig_i)  + noise_i

    bmap_nl_i = flt_.box_map(frame_nl_i, n=BOX_N, m=BOX_M)
    bmap_wl_i = flt_.box_map(frame_wl_i, n=BOX_N, m=BOX_M)
    resp_nl_i = flt_.matched(frame_nl_i - flt_.bias, tmpl_mc)
    resp_wl_i = flt_.matched(frame_wl_i - flt_.bias, tmpl_mc)

    snr['box_nl'].append(bmap_nl_i.max() / sigma_box_th)
    snr['box_wl'].append(bmap_wl_i.max() / sigma_box_th)
    snr['mf_nl' ].append(resp_nl_i.max() / sigma_mf_th)
    snr['mf_wl' ].append(resp_wl_i.max() / sigma_mf_th)

snr = {k: np.array(v) for k, v in snr.items()}

print(f"\n{'Filter':22s}  {'mean':>8}  {'std':>7}  {'min':>7}  {'max':>7}")
print('-' * 55)
for k, label in [('box_nl','Box  (no lag)'), ('box_wl','Box  (with lag)'),
                  ('mf_nl', 'MF   (no lag)'), ('mf_wl', 'MF   (with lag)')]:
    v = snr[k]
    print(f"{label:22s}  {v.mean():8.2f}  {v.std():7.2f}  {v.min():7.2f}  {v.max():7.2f}")

# ── 直方图 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
all_vals = np.concatenate(list(snr.values()))
bins = np.linspace(all_vals.min()*0.9, all_vals.max()*1.05, 35)

for ax, (k1, k2, title) in zip(axes, [
    ('box_nl', 'box_wl', f'Box {BOX_N}×{BOX_M}'),
    ('mf_nl',  'mf_wl',  f'MF lag  kernel {tmpl_mc.shape}')]):
    ax.hist(snr[k1], bins=bins, alpha=0.6, label=f'no-lag   μ={snr[k1].mean():.1f}')
    ax.hist(snr[k2], bins=bins, alpha=0.6, label=f'with-lag μ={snr[k2].mean():.1f}')
    ax.axvline(snr[k1].mean(), ls='--', lw=1.2)
    ax.axvline(snr[k2].mean(), ls='--', lw=1.2)
    ax.set_xlabel('SNR  (signal / σ_theory)')
    ax.set_ylabel('count')
    ax.set_title(f'{title}  N={N_MC}')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
plt.suptitle(f'A∈[{A_LO:.0f},{A_HI:.0f}] DN   jitter∈[−{JIT_MAX},{JIT_MAX}] fine-px   '
             f'RN={RN}  BIAS={BIAS}', fontsize=10)
plt.tight_layout()
plt.show()

MC kernel shape: (4, 24)  ||tmpl||=8.4319

Filter                      mean      std      min      max
-------------------------------------------------------
Box  (no lag)              10.41     2.83     4.82    17.00
Box  (with lag)             7.68     2.22     3.87    14.04
MF   (no lag)              12.38     3.54     5.49    21.09
MF   (with lag)             8.48     2.60     3.82    15.04
